# Neural Field Diffusion on ShapeNetCore.v2

This notebook trains the neural field diffusion model on real 3D shapes from ShapeNetCore.v2.

**Dataset**: [ShapeNetCore.v2](https://www.kaggle.com/datasets/hajareddagni/shapenetcorev2)
- ~51,300 unique 3D models
- 55 common object categories
- OBJ mesh format

**Key Differences from Toy Data**:
1. Real complex geometry (not parametric)
2. Large variety within categories
3. Need to sample point clouds from meshes
4. Requires larger model capacity

## Setup

1. Download ShapeNetCore.v2 from Kaggle
2. Extract to `data/shapenet/` directory
3. Run this notebook

In [ ]:
import sys
sys.path.append('..')

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm
from pathlib import Path
import time
import trimesh
import warnings
warnings.filterwarnings('ignore')

# Our modules
from src.models.neural_field import NeuralFieldDiffusion
from src.models.patched_neural_field import PatchedNeuralFieldDiffusion  # NEW
from src.models.sdf_field import SDFNeuralField, SDFFlowMatchingLoss
from src.diffusion.flow_matching import FlowMatchingLoss, FlowMatchingSampler

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Check for trimesh
try:
    import trimesh
    print(f"trimesh version: {trimesh.__version__}")
except ImportError:
    print("Please install trimesh: pip install trimesh[easy]")

## 1. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Data paths
SHAPENET_PATHS = [
    Path('../data/shapenet/ShapeNetCore.v2'),
    Path('../data/ShapeNetCore.v2'),
    Path.home() / '.cache/kagglehub/datasets/hajareddagni/shapenetcorev2/versions/1/ShapeNetCore.v2/ShapeNetCore.v2',
    Path('/home/idies/workspace/Temporary/dpark1/scratch/conda/conda_envs/mamba/.cache/kagglehub/datasets/hajareddagni/shapenetcorev2/versions/1/ShapeNetCore.v2/ShapeNetCore.v2'),
]

SHAPENET_ROOT = None
for path in SHAPENET_PATHS:
    if path.exists():
        synset_dirs = [d for d in path.iterdir() if d.is_dir() and d.name.isdigit()]
        if synset_dirs:
            SHAPENET_ROOT = path
            break

if SHAPENET_ROOT is None:
    SHAPENET_ROOT = SHAPENET_PATHS[0]
    print(f"WARNING: ShapeNet not auto-detected!")

CATEGORY_IDS = {
    'airplane': '02691156', 'car': '02958343', 'chair': '03001627',
    'table': '04379243', 'sofa': '04256520', 'lamp': '03636649',
    'vessel': '04530566', 'rifle': '04090263', 'speaker': '03691459',
    'bench': '02828884', 'cabinet': '02933112', 'display': '03211117',
    'phone': '04401088', 'watercraft': '04530566',
}

TRAIN_CATEGORIES = ['chair']

# Data settings
PATCH_SIZE = 16           # Points per patch (for patched model)
N_POINTS = 1024           # Must be divisible by PATCH_SIZE
MAX_SHAPES_PER_CAT = 500
NORMALIZE = True
AUGMENT = True

# Model type: 'velocity', 'sdf', or 'patched' (NEW!)
MODEL_TYPE = 'patched'  # <-- Try the new patched architecture

# Model config (smaller for testing)
HIDDEN_SIZE = 192
HIDDEN_SIZE_X = 48
NUM_HEADS = 4
NUM_BLOCKS = 6
NUM_NERF_BLOCKS = 3      # For patched model
NERF_MLP_RATIO = 2
MAX_FREQS = 6
BLEND_TEMPERATURE = 0.1  # For patched model distance weighting

# Training
EPOCHS = 200
BATCH_SIZE = 16
LR = 1e-4
GRADIENT_ACCUMULATION = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"Categories: {TRAIN_CATEGORIES}")
print(f"Model type: {MODEL_TYPE}")
print(f"N_POINTS: {N_POINTS}, PATCH_SIZE: {PATCH_SIZE} → {N_POINTS // PATCH_SIZE} patches")

if SHAPENET_ROOT.exists():
    synset_dirs = sorted([d.name for d in SHAPENET_ROOT.iterdir() if d.is_dir() and d.name.isdigit()])
    print(f"✓ ShapeNet found with {len(synset_dirs)} categories")
    for cat in TRAIN_CATEGORIES:
        synset = CATEGORY_IDS.get(cat, cat)
        if synset in synset_dirs:
            cat_path = SHAPENET_ROOT / synset
            n_shapes = len([d for d in cat_path.iterdir() if d.is_dir()])
            print(f"  ✓ {cat} ({synset}): {n_shapes} shapes")

## 2. ShapeNet Dataset

In [ ]:
class ShapeNetDataset(Dataset):
    """
    ShapeNetCore.v2 Dataset for point cloud generation.
    
    Loads meshes (OBJ or PLY) and samples point clouds from surfaces.
    """
    
    def __init__(self, root_dir, categories, n_points=2048, 
                 max_shapes_per_cat=None, normalize=True, augment=True,
                 cache_dir=None, debug=False):
        """
        Args:
            root_dir: Path to ShapeNetCore.v2
            categories: List of category names or synset IDs
            n_points: Number of points to sample per shape
            max_shapes_per_cat: Limit shapes per category (for testing)
            normalize: Whether to normalize to unit sphere
            augment: Whether to apply random rotation
            cache_dir: Directory to cache sampled point clouds
            debug: Print debug info about directory structure
        """
        self.root_dir = Path(root_dir)
        self.n_points = n_points
        self.normalize = normalize
        self.augment = augment
        self.cache_dir = Path(cache_dir) if cache_dir else None
        
        # Map category names to synset IDs
        self.synset_ids = []
        for cat in categories:
            if cat in CATEGORY_IDS:
                self.synset_ids.append(CATEGORY_IDS[cat])
            else:
                self.synset_ids.append(cat)  # Assume it's already a synset ID
        
        # Find all shape directories
        self.shape_paths = []
        self.shape_categories = []
        
        for synset_id in self.synset_ids:
            cat_dir = self.root_dir / synset_id
            if not cat_dir.exists():
                print(f"Warning: Category dir {cat_dir} not found")
                continue
            
            shape_dirs = sorted([d for d in cat_dir.iterdir() if d.is_dir()])
            
            if debug and shape_dirs:
                print(f"\nDebug: Category {synset_id}")
                print(f"  Found {len(shape_dirs)} shape directories")
                first_shape = shape_dirs[0]
                print(f"  First shape: {first_shape.name}")
                print(f"  Contents: {[x.name for x in first_shape.iterdir()]}")
                models_dir = first_shape / 'models'
                if models_dir.exists():
                    print(f"  models/ contents: {[x.name for x in models_dir.iterdir()]}")
                # Check for mesh files (OBJ or PLY)
                mesh_files = list(first_shape.rglob('*.obj')) + list(first_shape.rglob('*.ply'))
                print(f"  Mesh files found: {[str(f.relative_to(first_shape)) for f in mesh_files[:5]]}")
            
            if max_shapes_per_cat:
                shape_dirs = shape_dirs[:max_shapes_per_cat]
            
            for shape_dir in shape_dirs:
                model_path = None
                
                # Try common ShapeNet paths - BOTH OBJ AND PLY
                candidates = [
                    # PLY files (Kaggle version)
                    shape_dir / 'models' / 'model_normalized.ply',
                    shape_dir / 'models' / 'model.ply',
                    shape_dir / 'model_normalized.ply',
                    shape_dir / 'model.ply',
                    # OBJ files (standard version)
                    shape_dir / 'models' / 'model_normalized.obj',
                    shape_dir / 'models' / 'model.obj',
                    shape_dir / 'model_normalized.obj',
                    shape_dir / 'model.obj',
                ]
                
                for candidate in candidates:
                    if candidate.exists():
                        model_path = candidate
                        break
                
                # If not found, search for any mesh file
                if model_path is None:
                    mesh_files = list(shape_dir.rglob('*.ply')) + list(shape_dir.rglob('*.obj'))
                    if mesh_files:
                        model_path = mesh_files[0]
                
                if model_path is not None:
                    self.shape_paths.append(model_path)
                    self.shape_categories.append(synset_id)
        
        print(f"Found {len(self.shape_paths)} shapes across {len(self.synset_ids)} categories")
        
        if len(self.shape_paths) > 0:
            print(f"  Using format: {self.shape_paths[0].suffix}")
        
        # Setup cache
        if self.cache_dir:
            self.cache_dir.mkdir(parents=True, exist_ok=True)
    
    def __len__(self):
        return len(self.shape_paths)
    
    def _get_cache_path(self, idx):
        """Get cache file path for a shape."""
        if self.cache_dir is None:
            return None
        shape_id = self.shape_paths[idx].parent.parent.name
        cat_id = self.shape_categories[idx]
        return self.cache_dir / f"{cat_id}_{shape_id}_{self.n_points}.npy"
    
    def _load_and_sample(self, idx):
        """Load mesh and sample points."""
        # Check cache first
        cache_path = self._get_cache_path(idx)
        if cache_path and cache_path.exists():
            return np.load(cache_path)
        
        # Load mesh (trimesh handles both OBJ and PLY)
        mesh_path = self.shape_paths[idx]
        try:
            mesh = trimesh.load(mesh_path, force='mesh')
        except Exception as e:
            print(f"Error loading {mesh_path}: {e}")
            return np.random.randn(self.n_points, 3).astype(np.float32)
        
        # Handle scene vs mesh
        if isinstance(mesh, trimesh.Scene):
            meshes = [g for g in mesh.geometry.values() if isinstance(g, trimesh.Trimesh)]
            if meshes:
                mesh = trimesh.util.concatenate(meshes)
            else:
                return np.random.randn(self.n_points, 3).astype(np.float32)
        
        # Sample points from surface
        try:
            points, _ = trimesh.sample.sample_surface(mesh, self.n_points)
            points = points.astype(np.float32)
        except Exception as e:
            print(f"Error sampling {mesh_path}: {e}")
            return np.random.randn(self.n_points, 3).astype(np.float32)
        
        # Cache the result
        if cache_path:
            np.save(cache_path, points)
        
        return points
    
    def _normalize(self, points):
        """Normalize to unit sphere."""
        centroid = points.mean(axis=0)
        points = points - centroid
        max_dist = np.max(np.linalg.norm(points, axis=1))
        if max_dist > 0:
            points = points / max_dist
        return points
    
    def _random_rotate(self, points):
        """Apply random SO(3) rotation."""
        random_matrix = np.random.randn(3, 3)
        q, r = np.linalg.qr(random_matrix)
        if np.linalg.det(q) < 0:
            q[:, 0] *= -1
        return (points @ q.T).astype(np.float32)
    
    def __getitem__(self, idx):
        points = self._load_and_sample(idx)
        
        if self.normalize:
            points = self._normalize(points)
        
        if self.augment:
            points = self._random_rotate(points)
        
        return torch.tensor(points, dtype=torch.float32)
    
    def get_category(self, idx):
        """Get category of shape at index."""
        return self.shape_categories[idx]

In [ ]:
# Create dataset with debug=True to see directory structure
cache_dir = Path('../data/shapenet_cache')

dataset = ShapeNetDataset(
    root_dir=SHAPENET_ROOT,
    categories=TRAIN_CATEGORIES,
    n_points=N_POINTS,
    max_shapes_per_cat=MAX_SHAPES_PER_CAT,
    normalize=NORMALIZE,
    augment=AUGMENT,
    cache_dir=cache_dir,
    debug=True  # Enable to see what's inside the directories
)

if len(dataset) > 0:
    dataloader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )
    print(f"\nDataset size: {len(dataset)}")
    print(f"Batches per epoch: {len(dataloader)}")
else:
    print("\n❌ Dataset is empty - cannot create dataloader")
    print("Please check the directory structure above and update ShapeNetDataset accordingly")

In [ ]:
# Visualize some training samples
fig = plt.figure(figsize=(16, 8))

for i in range(8):
    sample = dataset[i].numpy()
    
    ax = fig.add_subplot(2, 4, i + 1, projection='3d')
    
    # Subsample for visualization
    vis_idx = np.random.choice(len(sample), min(1000, len(sample)), replace=False)
    vis_points = sample[vis_idx]
    
    colors = vis_points[:, 2]  # Color by z
    ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
               c=colors, cmap='viridis', s=1, alpha=0.6)
    ax.set_title(f'Shape {i+1}')
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([-1, 1])
    ax.view_init(elev=20, azim=45 + i*15)

plt.suptitle(f'ShapeNet {TRAIN_CATEGORIES} - Training Samples ({N_POINTS} points)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Model

In [ ]:
# Create model
if MODEL_TYPE == 'patched':
    # NEW: Patched architecture with space-filling curve serialization
    model = PatchedNeuralFieldDiffusion(
        in_channels=3,
        out_channels=3,
        hidden_size=HIDDEN_SIZE,
        hidden_size_x=HIDDEN_SIZE_X,
        num_heads=NUM_HEADS,
        num_cond_blocks=NUM_BLOCKS,      # DiT blocks (Stage 1) - renamed from num_blocks
        num_nerf_blocks=NUM_NERF_BLOCKS,  # NerfBlocks (Stage 2)
        nerf_mlp_ratio=NERF_MLP_RATIO,
        max_freqs=MAX_FREQS,
        patch_size=PATCH_SIZE,
        blend_temperature=BLEND_TEMPERATURE,
    ).to(DEVICE)
    loss_fn = FlowMatchingLoss(schedule_type='linear')
    
elif MODEL_TYPE == 'sdf':
    model = SDFNeuralField(
        in_channels=3,
        hidden_size=HIDDEN_SIZE,
        hidden_size_x=HIDDEN_SIZE_X,
        num_heads=NUM_HEADS,
        num_blocks=NUM_BLOCKS,
        num_cond_blocks=NUM_BLOCKS // 2,
        nerf_mlp_ratio=NERF_MLP_RATIO,
        max_freqs=MAX_FREQS,
    ).to(DEVICE)
    loss_fn = SDFFlowMatchingLoss(schedule_type='linear', eikonal_weight=0.0)
    
else:  # velocity
    model = NeuralFieldDiffusion(
        in_channels=3,
        out_channels=3,
        hidden_size=HIDDEN_SIZE,
        hidden_size_x=HIDDEN_SIZE_X,
        num_heads=NUM_HEADS,
        num_blocks=NUM_BLOCKS,
        num_cond_blocks=NUM_BLOCKS // 2,
        nerf_mlp_ratio=NERF_MLP_RATIO,
        max_freqs=MAX_FREQS,
    ).to(DEVICE)
    loss_fn = FlowMatchingLoss(schedule_type='linear')

sampler = FlowMatchingSampler(model)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model type: {MODEL_TYPE}")
print(f"Parameters: {n_params:,}")

if MODEL_TYPE == 'patched':
    print(f"Architecture: {NUM_BLOCKS} Transformer + {NUM_NERF_BLOCKS} PatchNerfBlocks")
    print(f"Patches: {N_POINTS // PATCH_SIZE} patches of {PATCH_SIZE} points each")
else:
    print(f"Architecture: DiT + NerfBlocks")

# Test forward pass
with torch.no_grad():
    test_input = torch.randn(2, N_POINTS, 3, device=DEVICE)
    test_t = torch.rand(2, device=DEVICE)
    test_output = model(test_input, test_t)
    print(f"\nTest - Input: {test_input.shape}, Output: {test_output.shape}")

## 4. Training

In [ ]:
# Setup optimizer and scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Mixed precision training (if available)
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler() if use_amp else None

# Training history
train_losses = []
epoch_times = []

print(f"Optimizer: AdamW (lr={LR})")
print(f"Scheduler: CosineAnnealing")
print(f"Mixed precision: {use_amp}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION}")

In [ ]:
def train_epoch(model, dataloader, optimizer, loss_fn, device, 
                scaler=None, grad_accum=1):
    """Train for one epoch with gradient accumulation."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(dataloader):
        x0 = batch.to(device)
        
        # Forward pass with mixed precision
        if scaler is not None:
            with torch.cuda.amp.autocast():
                output = loss_fn(model, x0)
                loss = output['loss'] / grad_accum
            scaler.scale(loss).backward()
        else:
            output = loss_fn(model, x0)
            loss = output['loss'] / grad_accum
            loss.backward()
        
        total_loss += output['loss'].item()
        n_batches += 1
        
        # Gradient accumulation step
        if (batch_idx + 1) % grad_accum == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad()
    
    return total_loss / n_batches


def generate_samples(model, sampler, n_samples=4, n_points=2048, n_steps=50,
                     method='sde', noise_scale=0.1, device='cpu'):
    """Generate samples."""
    model.eval()
    noise = torch.randn(n_samples, n_points, 3, device=device)
    
    with torch.no_grad():
        if method == 'euler':
            samples = sampler.sample_euler(noise, n_steps=n_steps)
        elif method == 'sde':
            samples = sampler.sample_sde(noise, n_steps=n_steps,
                                          noise_scale=noise_scale, decay='linear')
        else:
            samples = sampler.sample_euler(noise, n_steps=n_steps)
    
    return samples

In [ ]:
# Training loop
print(f"Training for {EPOCHS} epochs on {TRAIN_CATEGORIES}...")
print("="*60)

best_loss = float('inf')
pbar = tqdm(range(EPOCHS), desc="Training")

for epoch in pbar:
    start_time = time.time()
    
    loss = train_epoch(
        model, dataloader, optimizer, loss_fn, DEVICE,
        scaler=scaler, grad_accum=GRADIENT_ACCUMULATION
    )
    
    scheduler.step()
    epoch_time = time.time() - start_time
    
    train_losses.append(loss)
    epoch_times.append(epoch_time)
    
    current_lr = scheduler.get_last_lr()[0]
    pbar.set_postfix({'loss': f'{loss:.4f}', 'lr': f'{current_lr:.2e}'})
    
    # Logging
    if (epoch + 1) % 20 == 0:
        tqdm.write(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | LR: {current_lr:.2e} | Time: {epoch_time:.1f}s")
    
    # Save best model
    if loss < best_loss:
        best_loss = loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
        }, f'../experiments/outputs/shapenet_{"_".join(TRAIN_CATEGORIES)}_best.pt')

print("\n" + "="*60)
print(f"Training complete!")
print(f"Best loss: {best_loss:.4f}")
print(f"Average epoch time: {np.mean(epoch_times):.1f}s")

In [ ]:
# Plot training curve
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
if len(train_losses) > 10:
    plt.plot(train_losses[10:])
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss (after warmup)')
    plt.grid(True)

plt.tight_layout()
plt.show()

### Diagnostic: Check What's Going Wrong

In [ ]:
# === DIAGNOSTIC: Run this to understand what's happening ===

print("="*60)
print("DIAGNOSTIC ANALYSIS")
print("="*60)

# 1. Check final loss value
print(f"\n1. TRAINING LOSS:")
print(f"   Final loss: {train_losses[-1]:.4f}")
print(f"   Min loss: {min(train_losses):.4f}")
print(f"   Loss decreased: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}")

if train_losses[-1] > 0.3:
    print("   ⚠️  Loss > 0.3 indicates model is NOT learning well")
    print("   ⚠️  Expected: < 0.2 for good convergence")

# 2. Check output statistics
print(f"\n2. OUTPUT STATISTICS:")
model.eval()
with torch.no_grad():
    test_noise = torch.randn(4, N_POINTS, 3, device=DEVICE)
    test_t = torch.ones(4, device=DEVICE) * 0.5
    test_output = model(test_noise, test_t)
    
print(f"   Output mean: {test_output.mean().item():.6f}")
print(f"   Output std: {test_output.std().item():.6f}")
print(f"   Output range: [{test_output.min().item():.3f}, {test_output.max().item():.3f}]")

if test_output.std().item() < 0.1:
    print("   ⚠️  Low output variance - model may be collapsing")
if abs(test_output.mean().item()) < 1e-4 and test_output.std().item() < 0.01:
    print("   ❌ Output near zero - likely dead network or gradient issue")

# 3. Check if model learns identity at t=0
print(f"\n3. IDENTITY CHECK (at t=0, velocity should be ~0):")
with torch.no_grad():
    clean_data = dataset[0].unsqueeze(0).to(DEVICE)
    t_zero = torch.zeros(1, device=DEVICE)
    v_at_zero = model(clean_data, t_zero)
print(f"   Velocity at t=0: mean={v_at_zero.mean().item():.4f}, std={v_at_zero.std().item():.4f}")
if v_at_zero.std().item() > 0.5:
    print("   ⚠️  High velocity at t=0 - model hasn't learned the manifold")

# 4. Check data quality
print(f"\n4. DATA CHECK:")
sample = dataset[0].numpy()
print(f"   Point cloud range: [{sample.min():.3f}, {sample.max():.3f}]")
print(f"   Point cloud std: {sample.std():.3f}")
print(f"   Points near origin: {(np.abs(sample).max(axis=1) < 0.1).sum()}/{len(sample)}")

# 5. Gradient check
print(f"\n5. GRADIENT FLOW CHECK:")
model.train()
test_input = torch.randn(2, N_POINTS, 3, device=DEVICE, requires_grad=True)
test_t = torch.rand(2, device=DEVICE)
output = loss_fn(model, test_input)
output['loss'].backward()

zero_grad_layers = []
for name, param in model.named_parameters():
    if param.grad is not None and param.grad.abs().max() < 1e-8:
        zero_grad_layers.append(name)
        
if zero_grad_layers:
    print(f"   ⚠️  Zero gradients in: {zero_grad_layers[:5]}...")
else:
    print("   ✓ Gradients flowing to all layers")

# 6. Compare to GROUND TRUTH velocities
print(f"\n6. FLOW MATCHING TARGET CHECK:")
# What should the velocity be?
x0 = dataset[0].unsqueeze(0).to(DEVICE)  # Clean data
eps = torch.randn_like(x0)  # Noise
t = torch.tensor([0.5], device=DEVICE)
xt = (1 - t.view(-1, 1, 1)) * x0 + t.view(-1, 1, 1) * eps  # Noisy data
target_v = eps - x0  # Target velocity

with torch.no_grad():
    pred_v = model(xt, t)
    
mse = ((pred_v - target_v) ** 2).mean().item()
print(f"   Target velocity range: [{target_v.min().item():.3f}, {target_v.max().item():.3f}]")
print(f"   Pred velocity range: [{pred_v.min().item():.3f}, {pred_v.max().item():.3f}]")
print(f"   MSE between pred and target: {mse:.4f}")

if mse > 0.5:
    print("   ⚠️  High MSE - model not predicting correct velocities")

print("\n" + "="*60)
print("RECOMMENDATIONS:")
print("="*60)

if train_losses[-1] > 0.35:
    print("• Loss too high → Try:")
    print("  - Increase model size (hidden_size=256, num_blocks=8)")
    print("  - Train longer (500+ epochs)")
    print("  - Lower learning rate (1e-5)")
    print("  - Check if data is normalized correctly")
    
if test_output.std().item() < 0.1:
    print("• Output variance too low → Model may be ignoring input")
    print("  - Check patch_size (try 32 instead of 16)")
    print("  - Check blend_temperature (try 0.05 or 0.2)")

print("\n")

### Critical Test: Can the model overfit to ONE chair?

If the model can't overfit to a single shape, the architecture is broken.

In [ ]:
# === OVERFIT TEST: Train on SINGLE shape ===
# If this doesn't work, the architecture is fundamentally broken

print("="*60)
print("OVERFIT TEST: Training on ONE chair")
print("="*60)

# Get one chair
single_chair = dataset[0].unsqueeze(0).to(DEVICE)  # [1, N, 3]
print(f"Single chair shape: {single_chair.shape}")

# Create fresh small model
overfit_model = PatchedNeuralFieldDiffusion(
    in_channels=3, out_channels=3,
    hidden_size=128, hidden_size_x=32,
    num_heads=4, 
    num_cond_blocks=4,    # DiT blocks (Stage 1) - renamed from num_blocks
    num_nerf_blocks=2,    # NerfBlocks (Stage 2)
    nerf_mlp_ratio=2, max_freqs=6,
    patch_size=16, blend_temperature=0.1,
).to(DEVICE)

overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=1e-3)
overfit_loss_fn = FlowMatchingLoss(schedule_type='linear')

# Train for 500 steps on this ONE shape
overfit_losses = []
print("\nTraining on single chair (500 steps)...")

for step in range(500):
    overfit_optimizer.zero_grad()
    output = overfit_loss_fn(overfit_model, single_chair)
    loss = output['loss']
    loss.backward()
    overfit_optimizer.step()
    overfit_losses.append(loss.item())
    
    if step % 100 == 0:
        print(f"  Step {step}: loss = {loss.item():.4f}")

print(f"\nFinal overfit loss: {overfit_losses[-1]:.4f}")

# Generate from overfitted model
print("\nGenerating from overfitted model...")
overfit_model.eval()
overfit_sampler = FlowMatchingSampler(overfit_model)
noise = torch.randn(4, N_POINTS, 3, device=DEVICE)
with torch.no_grad():
    overfit_samples = overfit_sampler.sample_euler(noise, n_steps=100)
overfit_samples = overfit_samples.cpu().numpy()

# Visualize
fig = plt.figure(figsize=(16, 4))

# Ground truth
ax = fig.add_subplot(1, 5, 1, projection='3d')
gt = single_chair[0].cpu().numpy()
ax.scatter(gt[:, 0], gt[:, 1], gt[:, 2], c=gt[:, 2], cmap='viridis', s=1)
ax.set_title('Ground Truth')
ax.set_xlim([-1.2, 1.2]); ax.set_ylim([-1.2, 1.2]); ax.set_zlim([-1.2, 1.2])

# Generated samples
for i in range(4):
    ax = fig.add_subplot(1, 5, i + 2, projection='3d')
    gen = overfit_samples[i]
    ax.scatter(gen[:, 0], gen[:, 1], gen[:, 2], c=gen[:, 2], cmap='plasma', s=1)
    ax.set_title(f'Generated {i+1}')
    ax.set_xlim([-1.2, 1.2]); ax.set_ylim([-1.2, 1.2]); ax.set_zlim([-1.2, 1.2])

plt.suptitle(f'Overfit Test (loss={overfit_losses[-1]:.4f})', fontsize=14)
plt.tight_layout()
plt.show()

# Verdict
print("\n" + "="*60)
if overfit_losses[-1] < 0.1:
    print("✓ OVERFIT TEST PASSED - Architecture can learn")
    print("  Problem is likely: model capacity, data variety, or training time")
elif overfit_losses[-1] < 0.3:
    print("⚠️ PARTIAL SUCCESS - Loss reduced but not fully converged")
    print("  May need: more steps, lower LR, or architecture tweaks")
else:
    print("❌ OVERFIT TEST FAILED - Architecture cannot learn even ONE shape")
    print("  This indicates a fundamental bug in the architecture")
print("="*60)

## 5. Generate Samples

In [ ]:
# Generate samples
print("Generating samples...")

samples = generate_samples(
    model, sampler, 
    n_samples=8, 
    n_points=N_POINTS,
    n_steps=100,  # More steps for better quality
    method='sde',
    noise_scale=0.1,
    device=DEVICE
)
samples = samples.cpu().numpy()
print(f"Generated {samples.shape[0]} samples")

In [ ]:
# Compare GT vs Generated
fig = plt.figure(figsize=(16, 8))

# Ground truth
for i in range(4):
    gt = dataset[i].numpy()
    vis_idx = np.random.choice(len(gt), min(1000, len(gt)), replace=False)
    vis_points = gt[vis_idx]
    
    ax = fig.add_subplot(2, 4, i + 1, projection='3d')
    colors = vis_points[:, 2]
    ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
               c=colors, cmap='viridis', s=1, alpha=0.6)
    ax.set_title(f'GT {i+1}')
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])

# Generated
for i in range(4):
    gen = samples[i]
    vis_idx = np.random.choice(len(gen), min(1000, len(gen)), replace=False)
    vis_points = gen[vis_idx]
    
    ax = fig.add_subplot(2, 4, i + 5, projection='3d')
    colors = vis_points[:, 2]
    ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
               c=colors, cmap='plasma', s=1, alpha=0.6)
    ax.set_title(f'Generated {i+1}')
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])

plt.suptitle(f'ShapeNet {TRAIN_CATEGORIES}: GT (top) vs Generated (bottom)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Resolution Independence

In [ ]:
# Test resolution independence
print("Testing resolution independence...")

resolutions = [512, 1024, 2048, 4096, 8192]

fig = plt.figure(figsize=(20, 4))

for i, n_pts in enumerate(resolutions):
    noise = torch.randn(1, n_pts, 3, device=DEVICE)
    
    model.eval()
    with torch.no_grad():
        sample = sampler.sample_sde(noise, n_steps=100, noise_scale=0.1, decay='linear')
    
    sample = sample.cpu().numpy()[0]
    
    # Subsample for visualization
    vis_idx = np.random.choice(len(sample), min(1000, len(sample)), replace=False)
    vis_points = sample[vis_idx]
    
    ax = fig.add_subplot(1, 5, i + 1, projection='3d')
    colors = vis_points[:, 2]
    ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
               c=colors, cmap='viridis', s=1, alpha=0.6)
    ax.set_title(f'N = {n_pts}')
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])
    
    print(f"  Generated {n_pts:5d} points")

plt.suptitle('Resolution Independence on ShapeNet', fontsize=14)
plt.tight_layout()
plt.show()

## 6.1 Super-Resolution (Patched Model Only)

The patched architecture enables generation at arbitrary resolutions through:
- Morton curve serialization adapts to any N
- Per-patch neural fields blend smoothly via distance weighting
- Can upsample existing samples or generate directly at high resolution

In [ ]:
if MODEL_TYPE == 'patched':
    print("Testing super-resolution with patched model...")
    
    # Direct generation at multiple resolutions
    SUPERRES_POINTS = [1024, 2048, 4096, 8192, 16384]
    superres_samples = {}
    
    for n_pts in SUPERRES_POINTS:
        print(f"  Generating at {n_pts} points...", end=" ")
        start = time.time()
        
        noise = torch.randn(1, n_pts, 3, device=DEVICE)
        model.eval()
        with torch.no_grad():
            sample = sampler.sample_sde(noise, n_steps=100, noise_scale=0.1, decay='linear')
        superres_samples[n_pts] = sample.cpu().numpy()[0]
        
        print(f"done in {time.time() - start:.1f}s")
    
    # Visualize
    fig = plt.figure(figsize=(20, 4))
    
    for i, n_pts in enumerate(SUPERRES_POINTS):
        sample = superres_samples[n_pts]
        vis_idx = np.random.choice(len(sample), min(2000, len(sample)), replace=False)
        vis_points = sample[vis_idx]
        
        ax = fig.add_subplot(1, len(SUPERRES_POINTS), i + 1, projection='3d')
        colors = vis_points[:, 2]
        ax.scatter(vis_points[:, 0], vis_points[:, 1], vis_points[:, 2],
                   c=colors, cmap='viridis', s=max(0.5, 3 - i*0.5), alpha=0.6)
        ax.set_title(f'{n_pts} pts')
        ax.set_xlim([-1.2, 1.2])
        ax.set_ylim([-1.2, 1.2])
        ax.set_zlim([-1.2, 1.2])
    
    plt.suptitle(f'Super-Resolution on ShapeNet (trained at {N_POINTS} pts)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Super-resolution testing requires MODEL_TYPE='patched'")

In [ ]:
if MODEL_TYPE == 'patched':
    def upsample_shapenet(model, base_sample, n_points_target, n_refine_steps=30, device='cpu'):
        """Upsample a ShapeNet point cloud using patch context."""
        model.eval()
        
        x_base = torch.tensor(base_sample, dtype=torch.float32, device=device).unsqueeze(0)
        n_base = x_base.shape[1]
        
        with torch.no_grad():
            # Get context from base sample
            t_final = torch.tensor([0.0], device=device)
            patch_context, patch_centers = model.get_context(x_base, t_final)
            
            # Initialize with jittered copies
            n_extra = n_points_target - n_base
            if n_extra > 0:
                indices = torch.randint(0, n_base, (1, n_extra), device=device)
                extra = x_base[0, indices[0]] + 0.03 * torch.randn(n_extra, 3, device=device)
                x_target = torch.cat([x_base, extra.unsqueeze(0)], dim=1)
            else:
                x_target = x_base
            
            # Refine
            for step in range(n_refine_steps):
                t = torch.tensor([0.05 * (1 - step / n_refine_steps)], device=device)
                v = model.query_field(x_target, t, patch_context, patch_centers)
                x_target = x_target + v * 0.02
        
        return x_target[0].cpu().numpy()
    
    # Test upsampling on a generated sample
    print("\nTesting context-based upsampling...")
    base = superres_samples[1024]
    print(f"Base: {base.shape[0]} points")
    
    upsampled = {}
    upsampled[1024] = base
    for target in [2048, 4096, 8192]:
        print(f"  Upsampling to {target}...", end=" ")
        start = time.time()
        upsampled[target] = upsample_shapenet(model, base, target, device=DEVICE)
        print(f"done in {time.time() - start:.1f}s")
    
    # Compare direct vs upsampled
    fig = plt.figure(figsize=(16, 8))
    targets = [1024, 2048, 4096, 8192]
    
    for i, n_pts in enumerate(targets):
        # Direct generation
        direct = superres_samples.get(n_pts, superres_samples[1024])
        vis_idx = np.random.choice(len(direct), min(1500, len(direct)), replace=False)
        
        ax = fig.add_subplot(2, 4, i + 1, projection='3d')
        ax.scatter(direct[vis_idx, 0], direct[vis_idx, 1], direct[vis_idx, 2],
                   c=direct[vis_idx, 2], cmap='viridis', s=1, alpha=0.6)
        ax.set_title(f'Direct: {n_pts}')
        ax.set_xlim([-1.2, 1.2]); ax.set_ylim([-1.2, 1.2]); ax.set_zlim([-1.2, 1.2])
        
        # Upsampled
        up = upsampled[n_pts]
        vis_idx = np.random.choice(len(up), min(1500, len(up)), replace=False)
        
        ax = fig.add_subplot(2, 4, i + 5, projection='3d')
        ax.scatter(up[vis_idx, 0], up[vis_idx, 1], up[vis_idx, 2],
                   c=up[vis_idx, 2], cmap='plasma', s=1, alpha=0.6)
        ax.set_title(f'Upsampled: {n_pts}')
        ax.set_xlim([-1.2, 1.2]); ax.set_ylim([-1.2, 1.2]); ax.set_zlim([-1.2, 1.2])
    
    plt.suptitle('Direct Generation (top) vs Upsampling from 1024 (bottom)', fontsize=14)
    plt.tight_layout()
    plt.show()

## 7. Save Model

In [ ]:
# Save final checkpoint
categories_str = '_'.join(TRAIN_CATEGORIES)

# Build config based on model type
config = {
    'model_type': MODEL_TYPE,
    'hidden_size': HIDDEN_SIZE,
    'hidden_size_x': HIDDEN_SIZE_X,
    'num_heads': NUM_HEADS,
    'num_blocks': NUM_BLOCKS,
    'nerf_mlp_ratio': NERF_MLP_RATIO,
    'max_freqs': MAX_FREQS,
    'n_points': N_POINTS,
}

if MODEL_TYPE == 'patched':
    config.update({
        'num_nerf_blocks': NUM_NERF_BLOCKS,
        'patch_size': PATCH_SIZE,
        'blend_temperature': BLEND_TEMPERATURE,
    })
else:
    config['num_cond_blocks'] = NUM_BLOCKS // 2

checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': config,
    'train_losses': train_losses,
    'categories': TRAIN_CATEGORIES,
    'epochs': EPOCHS,
}

save_path = f'../experiments/outputs/shapenet_{categories_str}_{MODEL_TYPE}_final.pt'
torch.save(checkpoint, save_path)
print(f"Saved checkpoint to {save_path}")

## 8. Summary

### Training on ShapeNet

**Model Types Available**:
| Type | Architecture | Strengths |
|------|--------------|-----------|
| `velocity` | Global DiT + NerfBlocks | Simple, fast |
| `sdf` | SDF field + gradient velocity | Smooth, geometric |
| `patched` | Morton curve + per-patch NF | **Multi-modal, super-res** |

**Why Patched Model for ShapeNet**:
- Chairs have ~6 disconnected parts (legs, seat, back, arms)
- Per-patch context preserves local geometry
- Distance-weighted blending for smooth transitions
- Native super-resolution support (1024 → 16384 points)

**Key Differences from Toy Data**:
1. **More points**: 1024-2048 per shape
2. **Larger model**: ~2-5M params
3. **Gradient accumulation**: Effective batch size matters
4. **Mixed precision**: Essential for memory efficiency

**Recommendations**:
- Use `MODEL_TYPE = 'patched'` for complex geometry
- Start with single category (e.g., 'chair')
- Use caching for faster data loading
- Monitor loss - should decrease smoothly below 0.3

**Next Steps**:
1. Train longer (500+ epochs) for better quality
2. Try multiple categories
3. Add conditional generation
4. Evaluate with metrics (CD, EMD)